# 🗺️ Notebook 04 — Anatomy Atlas
## VERA: Visual Evidence–Report Alignment

This notebook builds and validates the chest anatomy atlas used by VERA.

**Steps:**
1. Define bounding boxes for 13 standard chest zones
2. Visualize atlas overlaid on sample CXR images
3. Test location string → zone mapping
4. Generate patch masks for the vision encoder grid
5. Export atlas for use in scoring

**⚡ This notebook runs on CPU and is independent of Notebooks 02-03.**

## 1. Setup

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (auto-detect Kaggle vs local)
if os.path.exists('/kaggle/working'):
    PROJECT_ROOT = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from config import PROCESSED_DIR, FIGURES_DIR, CHEST_ZONES, LOCATION_MAPPING, PATCH_GRID_CHEXAGENT, IS_KAGGLE
from src.anatomy_atlas import (
    location_to_zones, claim_to_region, zone_to_bbox,
    bbox_to_patch_mask, get_all_zone_masks, get_zone_colors,
    visualize_atlas_on_image, CHEST_ZONES as ATLAS_ZONES
)
from src.data_utils import load_json

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

## 2. Atlas Zone Definitions

We define 13 standard chest zones on a normalized [0,1] coordinate system.

**Convention:** (0,0) = top-left, (1,1) = bottom-right.  
In radiology: patient's right = image left (PA view).

In [ ]:
# Display all atlas zones
print("CHEST ANATOMY ATLAS — 13 Zones")
print("="*60)
print(f"{'Zone':<30} {'BBox (x1, y1, x2, y2)':<30}")
print("-"*60)
for zone, bbox in ATLAS_ZONES.items():
    print(f"{zone:<30} ({bbox[0]:.2f}, {bbox[1]:.2f}, {bbox[2]:.2f}, {bbox[3]:.2f})")

## 3. Visualize Atlas on Blank Canvas

In [ ]:
# Create atlas visualization on a blank canvas
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
colors = get_zone_colors()

# Draw zones as colored rectangles
for zone_name, bbox in ATLAS_ZONES.items():
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    color = np.array(colors[zone_name]) / 255.0
    
    rect = mpatches.FancyBboxPatch(
        (x1, y1), width, height,
        boxstyle="round,pad=0.005",
        facecolor=(*color, 0.4),
        edgecolor=(*color, 1.0),
        linewidth=2,
    )
    ax.add_patch(rect)
    
    # Label
    label = zone_name.replace('_', ' ').title()
    ax.text(
        x1 + width/2, y1 + height/2, label,
        ha='center', va='center', fontsize=7, fontweight='bold',
        color='black', bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7)
    )

ax.set_xlim(0, 1)
ax.set_ylim(1, 0)  # Invert y for image convention
ax.set_xlabel('X (normalized)', fontsize=12)
ax.set_ylabel('Y (normalized)', fontsize=12)
ax.set_title('VERA Chest Anatomy Atlas — 13 Zones', fontsize=16, fontweight='bold')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'atlas_zones.png'), dpi=200)
plt.show()
print(f"Saved: {FIGURES_DIR / 'atlas_zones.png'}")

## 4. Overlay Atlas on Real CXR Images

In [ ]:
# Load a sample CXR image
try:
    data = load_json(str(PROCESSED_DIR / 'test.json'))
    if data:
        sample_path = data[0]['image_path']
        sample_image = np.array(Image.open(sample_path).convert('RGB').resize((512, 512)))
        has_image = True
        print(f"Loaded sample image: {data[0]['image_id']}")
    else:
        has_image = False
except Exception as e:
    print(f"No dataset images available: {e}")
    print("Creating a synthetic test image...")
    sample_image = np.ones((512, 512, 3), dtype=np.uint8) * 200
    has_image = True

In [ ]:
if has_image:
    # Overlay atlas on image
    overlaid = visualize_atlas_on_image(sample_image, alpha=0.35)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    axes[0].imshow(sample_image)
    axes[0].set_title('Original CXR', fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(overlaid)
    axes[1].set_title('CXR with Atlas Overlay', fontsize=14)
    axes[1].axis('off')
    
    plt.suptitle('Anatomy Atlas Validation', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(FIGURES_DIR / 'atlas_overlay.png'), dpi=200)
    plt.show()
    
    print("\n🔍 CHECK: Do the colored zones roughly align with the anatomy?")
    print("   - Lung zones should cover the lung fields")
    print("   - Cardiac zone should cover the heart shadow")
    print("   - Mediastinum should be in the center")

## 5. Test Location String → Zone Mapping

In [ ]:
# Test various location strings
test_locations = [
    "right upper lobe",
    "left lower lobe",
    "bilateral lungs",
    "cardiac silhouette",
    "right hilum",
    "mediastinum",
    "right costophrenic angle",
    "lungs",
    "lower lobes",
    "right base",
    "left apex",
    "pleural",
    "retrocardiac",
    "perihilar",
    "aortic knob",
    "diaphragm",
    "unknown location",  # Should return empty
]

print("LOCATION MAPPING TEST")
print("="*70)
print(f"{'Input Location':<30} {'Mapped Zones':<40}")
print("-"*70)

for loc in test_locations:
    zones = location_to_zones(loc)
    zones_str = ', '.join(zones) if zones else '❌ NOT MAPPED'
    print(f"{loc:<30} {zones_str}")

## 6. Generate Patch Masks

In [ ]:
# Generate and visualize patch masks for the 24x24 grid
PATCH_GRID = PATCH_GRID_CHEXAGENT  # (24, 24)

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

zone_names = list(ATLAS_ZONES.keys())
for i, zone_name in enumerate(zone_names):
    if i >= 13:
        break
    
    mask = bbox_to_patch_mask(ATLAS_ZONES[zone_name], PATCH_GRID)
    axes[i].imshow(mask, cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    axes[i].set_title(zone_name.replace('_', '\n'), fontsize=9)
    axes[i].set_xlabel(f'({mask.sum():.0f} patches)', fontsize=8)
    axes[i].tick_params(labelsize=6)

# Hide empty subplots
for i in range(13, 15):
    axes[i].axis('off')

plt.suptitle(f'Patch Masks — {PATCH_GRID[0]}x{PATCH_GRID[1]} Grid', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'atlas_patch_masks.png'), dpi=150)
plt.show()

In [ ]:
# Test claim_to_region with combined masks
test_claims = [
    ("right lower lobe", "opacity"),
    ("bilateral lungs", "clear"),
    ("cardiac", "cardiomegaly"),
    ("left costophrenic angle", "effusion"),
    ("lower lobes", "infiltrates"),
]

fig, axes = plt.subplots(1, len(test_claims), figsize=(4*len(test_claims), 4))

for i, (location, finding) in enumerate(test_claims):
    mask = claim_to_region(location, PATCH_GRID)
    if mask is not None:
        axes[i].imshow(mask, cmap='Greens', vmin=0, vmax=1, interpolation='nearest')
        axes[i].set_title(f'"{finding}"\nin "{location}"', fontsize=10)
        axes[i].set_xlabel(f'{mask.sum():.0f} patches', fontsize=9)
    else:
        axes[i].text(0.5, 0.5, 'NOT MAPPED', ha='center', va='center')
        axes[i].set_title(f'"{location}"', fontsize=10)

plt.suptitle('Combined Region Masks for Claims', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'atlas_claim_masks.png'), dpi=150)
plt.show()

## 7. Export Atlas

In [ ]:
# Export atlas as JSON for reference
import json

atlas_export = {
    'zones': {k: list(v) for k, v in ATLAS_ZONES.items()},
    'location_mapping': {k: v for k, v in LOCATION_MAPPING.items()},
    'patch_grid': list(PATCH_GRID),
    'num_zones': len(ATLAS_ZONES),
    'coordinate_system': '(0,0)=top-left, (1,1)=bottom-right, PA view convention',
}

atlas_path = PROCESSED_DIR / 'anatomy_atlas.json'
with open(atlas_path, 'w') as f:
    json.dump(atlas_export, f, indent=2)

print(f"✅ Atlas exported to: {atlas_path}")
print(f"   Zones: {atlas_export['num_zones']}")
print(f"   Patch grid: {PATCH_GRID}")
print(f"   Location mappings: {len(atlas_export['location_mapping'])}")
print(f"\nNext: Run 05_vera_scoring.ipynb")